# Worksheet support: Working with Data (Advanced)
Prepared for Inuka Hulathduwage. The Series uses his four supplied course/grade pairs; the separate join exercise uses invented example students. All six supplied personal Explain It responses appear in the worksheet. Run all cells from this folder. Timing measurements are illustrative and machine dependent.

In [1]:
from pathlib import Path
import pandas as pd
BASE = Path('..')
OUT = BASE / 'Worksheet-OutputFiles'
OUT.mkdir(exist_ok=True)
titanic = pd.read_csv(BASE / 'titanic.csv')
students = pd.read_csv(BASE / 'students.csv')
pokemon = pd.read_csv(BASE / 'pokemon.csv')
ufo = pd.read_csv(BASE / 'ufo.csv', low_memory=False)
print('Loaded all four local datasets.')


Loaded all four local datasets.


In [2]:
# Series Basics: course names and numeric grades supplied by Inuka.
grades = pd.Series({
    'Advanced Data Structures': 96,
    'Programming Techniques': 70,
    'Network Security': 85,
    'Linear Algebra': 90,
}, name='Grade')
print(grades)
print('Average grade:', grades.mean())
assert grades.mean() == 85.25


Advanced Data Structures    96
Programming Techniques      70
Network Security            85
Linear Algebra              90
Name: Grade, dtype: int64
Average grade: 85.25


In [3]:
# First Look at a Dataset
print('Shape:', pokemon.shape)
print('Columns:', pokemon.columns.tolist())
print(pokemon.select_dtypes(include='number').describe().to_string())
missing = pokemon.isnull().sum()
print('Missing values:', missing.to_dict())
print('Most missing:', missing.idxmax(), int(missing.max()))
pokemon.info()


Shape: (800, 13)
Columns: ['#', 'Name', 'Type 1', 'Type 2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']
                #      Total          HP      Attack     Defense     Sp. Atk     Sp. Def       Speed  Generation
count  800.000000  800.00000  800.000000  800.000000  800.000000  800.000000  800.000000  800.000000   800.00000
mean   362.813750  435.10250   69.258750   79.001250   73.842500   72.820000   71.902500   68.277500     3.32375
std    208.343798  119.96304   25.534669   32.457366   31.183501   32.722294   27.828916   29.060474     1.66129
min      1.000000  180.00000    1.000000    5.000000    5.000000   10.000000   20.000000    5.000000     1.00000
25%    184.750000  330.00000   50.000000   55.000000   50.000000   49.750000   50.000000   45.000000     2.00000
50%    364.500000  450.00000   65.000000   75.000000   70.000000   65.000000   70.000000   65.000000     3.00000
75%    539.250000  515.00000   80.000000  100.000000   

In [4]:
# Titanic Slicing
third_class_survivors = titanic.loc[
    (titanic['Pclass'] == 3) & (titanic['Survived'] == 1),
    ['Name', 'Age', 'Fare'],
].sort_values('Fare', ascending=False)
print(third_class_survivors.to_string(index=False))
third_class_survivors.to_csv(OUT / 'third_class_survivors.csv', index=False)
assert third_class_survivors['Fare'].is_monotonic_decreasing


                                                     Name   Age    Fare
                                            Bing, Mr. Lee 32.00 56.4958
                                          Chip, Mr. Chang 32.00 56.4958
                                           Lang, Mr. Fang 26.00 56.4958
                                          Foo, Mr. Choong   NaN 56.4958
                                             Lam, Mr. Ali   NaN 56.4958
Asplund, Mrs. Carl Oscar (Selma Augusta Emilia Johansson) 38.00 31.3875
                        Asplund, Master. Edvin Rojj Felix  3.00 31.3875
                           Asplund, Miss. Lillian Gertrud  5.00 31.3875
                                      Moran, Miss. Bertha   NaN 24.1500
                                       McCoy, Mr. Bernard   NaN 23.2500
                                       McCoy, Miss. Agnes   NaN 23.2500
                   Peter, Mrs. Catherine (Catherine Rizk)   NaN 22.3583
                                        Peter, Miss. Anna   NaN 

In [5]:
# Clean a Column
missing_ufo = ufo.isnull().sum()
print(missing_ufo.to_string())
most_missing = missing_ufo.idxmax()
clean_ufo = ufo.copy()
# Country is categorical. Keep sightings with unknown country for overall
# reporting; use an explicit category rather than guessing a country or
# discarding records. Exclude Unknown when computing actual country rates.
clean_ufo[most_missing] = clean_ufo[most_missing].fillna('Unknown')
print('Filled column:', most_missing)
print('Filled entries:', int(missing_ufo[most_missing]))
assert clean_ufo[most_missing].isna().sum() == 0
assert len(clean_ufo) == len(ufo)
pd.DataFrame({'missing_before': missing_ufo,
              'missing_after': clean_ufo.isnull().sum()}).to_csv(OUT / 'ufo_missing_audit.csv')


date_time                        0
city_area                        0
state                         5797
country                       9670
ufo_shape                     1932
encounter_length                 3
described_encounter_length       0
description                     15
date_documented                  0
latitude                         1
longitude                        0
Filled column: country
Filled entries: 9670


In [6]:
# Build a Feature
students['total_score'] = students['math_score'] + students['science_score']
top_three = students.sort_values('total_score', ascending=False)[['first_name', 'total_score']].head(3)
print(top_three.to_string(index=False))
top_three.to_csv(OUT / 'top_three_students.csv', index=False)


 first_name  total_score
Christopher          197
   Danielle          195
    Bridget          192


In [7]:
# Loop vs. Vectorized
loop_totals = [row.math_score + row.science_score for row in students.itertuples(index=False)]
vector_totals = students['math_score'] + students['science_score']
assert loop_totals == vector_totals.tolist()
print('All', len(loop_totals), 'totals match.')
%timeit -r 3 -n 100 [row.math_score + row.science_score for row in students.itertuples(index=False)]
%timeit -r 3 -n 100 students['math_score'] + students['science_score']


All 50 totals match.
229 μs ± 4.06 μs per loop (mean ± std. dev. of 3 runs, 100 loops each)
57 μs ± 2.07 μs per loop (mean ± std. dev. of 3 runs, 100 loops each)


In [8]:
# Survival by Group
print('Survival by class:')
print(titanic.groupby('Pclass')['Survived'].mean().to_string())
print('Survival by class and sex:')
print(titanic.groupby(['Pclass', 'Sex'])['Survived'].mean().to_string())
# Keep passengers with missing Embarked as an explicit NaN group.
port_summary = (titanic.groupby('Embarked', dropna=False)
    .agg(passengers=('PassengerId', 'size'), average_Fare=('Fare', 'mean'),
         survival_rate=('Survived', 'mean'))
    .reset_index().sort_values('survival_rate', ascending=False))
print(port_summary.to_string(index=False))
assert port_summary['passengers'].sum() == len(titanic)
port_summary.to_csv(OUT / 'survival_by_port.csv', index=False)


Survival by class:
Pclass
1    0.629630
2    0.472826
3    0.242363
Survival by class and sex:
Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Embarked  passengers  average_Fare  survival_rate
     NaN           2     80.000000       1.000000
       C         168     59.954144       0.553571
       Q          77     13.276030       0.389610
       S         644     27.079812       0.336957


In [9]:
# Join Two Tables: invented students and grades, not personal course grades.
student_table = pd.DataFrame([
    {'student_id': 1, 'name': 'Sample Alex'},
    {'student_id': 2, 'name': 'Sample Blair'},
    {'student_id': 3, 'name': 'Sample Casey'},
    {'student_id': 4, 'name': 'Sample Drew'},
    {'student_id': 5, 'name': 'Sample Emery'},
])
grade_table = pd.DataFrame({'student_id': [1, 2, 3, 4], 'grade': [85, 92, 78, 88]})
left_result = student_table.merge(grade_table, on='student_id', how='left', validate='one_to_one')
inner_result = student_table.merge(grade_table, on='student_id', how='inner', validate='one_to_one')
print('LEFT:', len(left_result), 'rows; preserves all five students.')
print(left_result.to_string(index=False))
print('INNER:', len(inner_result), 'rows; only four students have matching grades.')
print(inner_result.to_string(index=False))
assert len(left_result) == 5 and len(inner_result) == 4
left_result.to_csv(OUT / 'sample_students_left_join.csv', index=False)
inner_result.to_csv(OUT / 'sample_students_inner_join.csv', index=False)


LEFT: 5 rows; preserves all five students.
 student_id         name  grade
          1  Sample Alex   85.0
          2 Sample Blair   92.0
          3 Sample Casey   78.0
          4  Sample Drew   88.0
          5 Sample Emery    NaN
INNER: 4 rows; only four students have matching grades.
 student_id         name  grade
          1  Sample Alex     85
          2 Sample Blair     92
          3 Sample Casey     78
          4  Sample Drew     88


In [10]:
# Additional worksheet concepts: positional selection, isin, and a sentinel.
print('First two passenger records:')
print(titanic.iloc[:2].to_string(index=False))
print('Passengers in classes 1 or 2:', len(titanic.loc[titanic['Pclass'].isin([1, 2])]))
survey = pd.DataFrame({'Age': [25, -1, 35]})
survey['Age'] = survey['Age'].replace(-1, float('nan'))
assert survey['Age'].isna().sum() == 1
print('Sentinel replacement:', survey['Age'].tolist())


First two passenger records:
 PassengerId  Survived  Pclass                                                Name    Sex  Age  SibSp  Parch    Ticket    Fare Cabin Embarked
           1         0       3                             Braund, Mr. Owen Harris   male 22.0      1      0 A/5 21171  7.2500   NaN        S
           2         1       1 Cumings, Mrs. John Bradley (Florence Briggs Thayer) female 38.0      1      0  PC 17599 71.2833   C85        C
Passengers in classes 1 or 2: 400
Sentinel replacement: [25.0, nan, 35.0]
